# ETL — Preparación y Carga de Datos
## Práctica 1 · SOG2 2S26 · Grupo 8

**Punto 1 — Preparación de datos:**
1. Extraer los datos del archivo `.csv`
2. Verificar si hay valores faltantes o duplicados y decidir cómo manejarlos
3. Asegurarse de que los tipos de datos sean correctos para cada columna
4. Cargar los datos a una base de datos SQL en la nube

**Estrategia de carga:** Incremental (soporta múltiples ejecuciones y múltiples archivos CSV)

---
## 1. Instalación de dependencias

In [1]:
# Ejecutar solo la primera vez (o en Google Colab)
!pip install pandas sqlalchemy psycopg2-binary python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 11.4 MB/s eta 0:00:00


## 2. Importar librerías

In [2]:
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
from sqlalchemy import create_engine, text, inspect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"pandas: {pd.__version__}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

pandas: 2.2.3
Fecha de ejecución: 2026-08-18 20:31:33


---
## 3. Extracción — Cargar el CSV

El archivo `Venta_online_c.csv` usa **punto y coma (`;`)** como separador, no coma.

In [5]:
#@title Seleccionar Origen de Datos { run: "auto" }
METODO_ORIGEN = "Subir archivos desde mi equipo" #@param ["Subir archivos desde mi equipo", "Ruta de archivo(s) local(es)"]
RUTAS_LOCALES = "/content/dataset_vuelos_crudo.csv" #@param {type:"string"}

archivos_a_procesar = []

if METODO_ORIGEN == "Subir archivos desde mi equipo":
    from google.colab import files
    print("Por favor selecciona los archivos (.csv o .xlsx) a subir:")
    uploaded = files.upload()
    archivos_a_procesar = list(uploaded.keys())
    print(f"Se subieron {len(archivos_a_procesar)} archivo(s): {archivos_a_procesar}")

else:
    # Rutas separadas por comas
    archivos_a_procesar = [p.strip() for p in RUTAS_LOCALES.split(",") if p.strip()]
    print(f"Archivos indicados: {archivos_a_procesar}")

# Carga de archivos a DataFrame
dfs = []
for path in archivos_a_procesar:
    if path.endswith(".csv"):
        # Mantenemos el separador ';' especificado en el notebook original
        df_temp = pd.read_csv(path, sep=';', encoding="utf-8")
    elif path.endswith( (".xlsx", ".xls") ):
        df_temp = pd.read_excel(path)
    else:
        print(f"Formato ignorado: {path}")
        continue
    print(f" '{path}': {df_temp.shape[0]} filas, {df_temp.shape[1]} columnas.")
    dfs.append(df_temp)

if dfs:
    df_raw = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal de registros crudos extraídos: {len(df_raw)}")
    display(df_raw.head(10))
else:
    raise ValueError("No se encontraron archivos válidos para procesar.")

Por favor selecciona los archivos (.csv o .xlsx) a subir:


Saving Venta_online_c.csv to Venta_online_c.csv
Se subieron 1 archivo(s): ['Venta_online_c.csv']
 'Venta_online_c.csv': 6500 filas, 12 columnas.

Total de registros crudos extraídos: 6500


,Id_cliente,Edad,Genero,Venta_total,N_Compras,FechaCompra,MontoCompra,MetodoPago,Tiempo,Navegador,Boletin,Vale
0,504308,18,1,737.4000,8,02.02.21,109.0540,2,1022,0,1,0
1,504309,46,1,689.2000,17,10.10.21,35.3260,1,865,0,0,0
2,504310,38,0,95.6000,1,27.08.21,89.3530,2,900,2,1,1
3,504311,44,1,75.7000,2,10.06.21,27.2200,1,793,0,1,0
4,504312,41,0,46.0000,2,08.06.21,28.5310,0,729,0,0,0
5,504313,24,1,20.9000,1,10.11.21,16.6110,1,878,2,0,1
6,504314,18,1,269.0000,6,01.02.21,44.6610,1,807,1,1,0
7,504315,27,0,376.5000,10,12.09.21,54.9770,2,689,3,1,1
8,504316,43,0,122.3000,4,15.03.21,42.8390,1,525,0,0,1
9,504317,44,0,239.4000,5,04.02.21,43.8830,1,932,1,1,0


---
## 4. Verificación de calidad de datos

Verificar si hay valores faltantes o duplicados y decidir cómo manejarlos.

In [6]:
print("="*60)
print("4.1 TIPOS DE DATOS ORIGINALES")
print("="*60)
print(df_raw.dtypes)
print()

4.1 TIPOS DE DATOS ORIGINALES
Id_cliente       int64
Edad             int64
Genero           int64
Venta_total    float64
N_Compras        int64
FechaCompra     object
MontoCompra    float64
MetodoPago       int64
Tiempo           int64
Navegador        int64
Boletin          int64
Vale             int64
dtype: object



In [7]:
print("="*60)
print("4.2 VALORES NULOS POR COLUMNA")
print("="*60)
nulos = df_raw.isnull().sum()
print(nulos)
print(f"\nTotal de valores nulos en el dataset: {nulos.sum()}")

if nulos.sum() == 0:
    print("\nNo se encontraron valores nulos. No se requiere imputación.")
else:
    print("\nSe encontraron valores nulos. Requiere tratamiento.")

4.2 VALORES NULOS POR COLUMNA
Id_cliente     0
Edad           0
Genero         0
Venta_total    0
N_Compras      0
FechaCompra    0
MontoCompra    0
MetodoPago     0
Tiempo         0
Navegador      0
Boletin        0
Vale           0
dtype: int64

Total de valores nulos en el dataset: 0

No se encontraron valores nulos. No se requiere imputación.


In [8]:
print("="*60)
print("4.3 FILAS DUPLICADAS")
print("="*60)
duplicadas = df_raw.duplicated().sum()
print(f"Filas completamente duplicadas: {duplicadas}")

dup_id = df_raw['Id_cliente'].duplicated().sum()
print(f"Id_cliente duplicados: {dup_id}")

if duplicadas == 0 and dup_id == 0:
    print("\nNo hay filas duplicadas ni Id_cliente repetidos.")
else:
    print("\nSe encontraron duplicados. Se procederá a eliminarlos.")
    df_raw = df_raw.drop_duplicates()
    df_raw = df_raw.drop_duplicates(subset=['Id_cliente'], keep='first')
    print(f"   Filas después de limpiar: {len(df_raw):,}")

4.3 FILAS DUPLICADAS
Filas completamente duplicadas: 0
Id_cliente duplicados: 0

No hay filas duplicadas ni Id_cliente repetidos.


In [9]:
print("="*60)
print("4.4 ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
df_raw.describe()

4.4 ESTADÍSTICAS DESCRIPTIVAS


,Id_cliente,Edad,Genero,Venta_total,N_Compras,MontoCompra,MetodoPago,Tiempo,Navegador,Boletin,Vale
count,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000,6500.0000
mean,507557.5000,36.3052,0.4812,206.2424,5.0900,39.7871,1.0398,767.3762,0.8823,0.4494,0.1929
std,1876.5327,11.3623,0.4997,215.5516,3.9550,19.5229,0.6401,181.7490,1.1565,0.4975,0.3946
min,504308.0000,18.0000,0.0000,9.0000,1.0000,7.2410,0.0000,180.0000,0.0000,0.0000,0.0000
25%,505932.7500,28.0000,0.0000,69.1000,2.0000,26.2238,1.0000,645.0000,0.0000,0.0000,0.0000
50%,507557.5000,36.0000,0.0000,137.3500,4.0000,35.7640,1.0000,768.0000,0.0000,0.0000,0.0000
75%,509182.2500,44.0000,1.0000,266.6000,7.0000,48.5598,1.0000,886.0000,2.0000,1.0000,0.0000
max,510807.0000,79.0000,1.0000,3169.0000,25.0000,199.3490,2.0000,1443.0000,4.0000,1.0000,1.0000


In [10]:
print("="*60)
print("4.5 VALORES ÚNICOS EN CAMPOS CATEGÓRICOS")
print("="*60)

categoricos = ['Genero', 'MetodoPago', 'Navegador', 'Boletin', 'Vale']

for col in categoricos:
    valores = sorted(df_raw[col].unique())
    print(f"  {col}: {valores}  (n_unicos = {len(valores)})")

# Verificar que los valores estén dentro de los rangos esperados según el PDF
assert set(df_raw['Genero'].unique()).issubset({0, 1}), "Genero tiene valores fuera de {0, 1}"
assert set(df_raw['MetodoPago'].unique()).issubset({0, 1, 2}), "MetodoPago tiene valores fuera de {0, 1, 2}"
assert set(df_raw['Navegador'].unique()).issubset({0, 1, 2, 3, 4}), "Navegador tiene valores fuera de {0..4}"
assert set(df_raw['Boletin'].unique()).issubset({0, 1}), "Boletin tiene valores fuera de {0, 1}"
assert set(df_raw['Vale'].unique()).issubset({0, 1}), "Vale tiene valores fuera de {0, 1}"

print("\nTodos los campos categóricos tienen valores válidos.")

4.5 VALORES ÚNICOS EN CAMPOS CATEGÓRICOS
  Genero: [np.int64(0), np.int64(1)]  (n_unicos = 2)
  MetodoPago: [np.int64(0), np.int64(1), np.int64(2)]  (n_unicos = 3)
  Navegador: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]  (n_unicos = 5)
  Boletin: [np.int64(0), np.int64(1)]  (n_unicos = 2)
  Vale: [np.int64(0), np.int64(1)]  (n_unicos = 2)

Todos los campos categóricos tienen valores válidos.


In [11]:
print("="*60)
print("4.6 VERIFICACIÓN DE RANGOS NUMÉRICOS")
print("="*60)

print(f"  Edad:        min={df_raw['Edad'].min()}, max={df_raw['Edad'].max()}")
print(f"  Venta_total: min={df_raw['Venta_total'].min()}, max={df_raw['Venta_total'].max()}")
print(f"  N_Compras:   min={df_raw['N_Compras'].min()}, max={df_raw['N_Compras'].max()}")
print(f"  MontoCompra: min={df_raw['MontoCompra'].min()}, max={df_raw['MontoCompra'].max()}")
print(f"  Tiempo:      min={df_raw['Tiempo'].min()}, max={df_raw['Tiempo'].max()}")

# Verificar que no hay valores negativos en campos que no deberían
assert (df_raw['Edad'] > 0).all(), "Hay edades <= 0"
assert (df_raw['Venta_total'] >= 0).all(), "Hay ventas totales negativas"
assert (df_raw['N_Compras'] > 0).all(), "Hay número de compras <= 0"
assert (df_raw['MontoCompra'] >= 0).all(), "Hay montos de compra negativos"
assert (df_raw['Tiempo'] > 0).all(), "Hay tiempos <= 0"

print("\nTodos los rangos numéricos son coherentes y sin valores anómalos.")

4.6 VERIFICACIÓN DE RANGOS NUMÉRICOS
  Edad:        min=18, max=79
  Venta_total: min=9.0, max=3169.0
  N_Compras:   min=1, max=25
  MontoCompra: min=7.241, max=199.349
  Tiempo:      min=180, max=1443

Todos los rangos numéricos son coherentes y sin valores anómalos.


### Resumen de verificación de calidad

| Verificación | Resultado | Acción |
|---|---|---|
| Valores nulos | 0 nulos | No se requiere imputación |
| Filas duplicadas | 0 duplicadas | No se requiere eliminación |
| Id_cliente duplicados | 0 duplicados | Cada cliente es único |
| Valores categóricos | Dentro de rango | Validados contra el enunciado |
| Valores numéricos | Coherentes | Sin negativos ni anómalos |

---
## 5. Transformación — Corrección de tipos de datos

Asegurarse de que los tipos de datos sean correctos para cada columna.

In [12]:
# Trabajamos sobre una copia para preservar el original
df = df_raw.copy()

# ============================================================
# 5.1 Convertir FechaCompra de texto a fecha
# Formato en CSV: DD.MM.AA (ej: 02.02.21 = 2 de febrero de 2021)
# ============================================================
df['FechaCompra'] = pd.to_datetime(df['FechaCompra'], format='%d.%m.%y')

print("Conversión de FechaCompra:")
print(f"  Fecha mínima: {df['FechaCompra'].min()}")
print(f"  Fecha máxima: {df['FechaCompra'].max()}")

# Validar que todas las fechas estén en 2021
assert df['FechaCompra'].dt.year.unique().tolist() == [2021], "Hay fechas fuera de 2021"
print("Todas las fechas corresponden al año 2021")

Conversión de FechaCompra:
  Fecha mínima: 2021-01-01 00:00:00
  Fecha máxima: 2021-12-31 00:00:00
Todas las fechas corresponden al año 2021


In [13]:
# ============================================================
# 5.2 Convertir tipos enteros y booleanos
# ============================================================
for col in ['Genero', 'MetodoPago', 'Navegador']:
    df[col] = df[col].astype('int8')

df['Boletin'] = df['Boletin'].astype(bool)
df['Vale'] = df['Vale'].astype(bool)

# ============================================================
# 5.3 Asegurar 4 decimales en campos monetarios
# ============================================================
df['Venta_total'] = df['Venta_total'].round(4)
df['MontoCompra'] = df['MontoCompra'].round(4)

print("Tipos de datos después de la conversión:")
print(df.dtypes)
print("\nTipos de datos corregidos exitosamente.")

Tipos de datos después de la conversión:
Id_cliente              int64
Edad                    int64
Genero                   int8
Venta_total           float64
N_Compras               int64
FechaCompra    datetime64[ns]
MontoCompra           float64
MetodoPago               int8
Tiempo                  int64
Navegador                int8
Boletin                  bool
Vale                     bool
dtype: object

Tipos de datos corregidos exitosamente.


In [14]:
# ============================================================
# 5.4 Renombrar columnas a snake_case
# ============================================================
df = df.rename(columns={
    'Id_cliente':  'id_cliente',
    'Edad':        'edad',
    'Genero':      'genero',
    'Venta_total': 'venta_total',
    'N_Compras':   'n_compras',
    'FechaCompra': 'fecha_compra',
    'MontoCompra': 'monto_compra',
    'MetodoPago':  'metodo_pago',
    'Tiempo':      'tiempo',
    'Navegador':   'navegador',
    'Boletin':     'boletin',
    'Vale':        'vale'
})

print("Columnas renombradas:")
print(list(df.columns))
df.head()

Columnas renombradas:
['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras', 'fecha_compra', 'monto_compra', 'metodo_pago', 'tiempo', 'navegador', 'boletin', 'vale']


,id_cliente,edad,genero,venta_total,n_compras,fecha_compra,monto_compra,metodo_pago,tiempo,navegador,boletin,vale
0,504308,18,1,737.4000,8,2021-02-02,109.0540,2,1022,0,True,False
1,504309,46,1,689.2000,17,2021-10-10,35.3260,1,865,0,False,False
2,504310,38,0,95.6000,1,2021-08-27,89.3530,2,900,2,True,True
3,504311,44,1,75.7000,2,2021-06-10,27.2200,1,793,0,True,False
4,504312,41,0,46.0000,2,2021-06-08,28.5310,0,729,0,False,False


---
## 6. Normalización — Separar en tablas relacionales

Se separan los datos en dos tablas:
- **`clientes`**: perfil del cliente con datos agregados del año (id, edad, género, venta total, número de compras)
- **`compras`**: detalle de compra individual (fecha, monto, método de pago, tiempo, navegador, boletín, vale)

In [15]:
# ============================================================
# Tabla clientes: perfil y métricas agregadas anuales
# ============================================================
df_clientes = df[['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras']].copy()

print(f"Tabla CLIENTES: {df_clientes.shape[0]:,} filas x {df_clientes.shape[1]} columnas")
df_clientes.head(10)

Tabla CLIENTES: 6,500 filas x 5 columnas


,id_cliente,edad,genero,venta_total,n_compras
0,504308,18,1,737.4000,8
1,504309,46,1,689.2000,17
2,504310,38,0,95.6000,1
3,504311,44,1,75.7000,2
4,504312,41,0,46.0000,2
5,504313,24,1,20.9000,1
6,504314,18,1,269.0000,6
7,504315,27,0,376.5000,10
8,504316,43,0,122.3000,4
9,504317,44,0,239.4000,5


In [16]:
# ============================================================
# Tabla compras: detalle de cada compra individual
# ============================================================
df_compras = df[['id_cliente', 'fecha_compra', 'monto_compra', 'metodo_pago',
                 'tiempo', 'navegador', 'boletin', 'vale']].copy()

print(f"Tabla COMPRAS: {df_compras.shape[0]:,} filas x {df_compras.shape[1]} columnas")
df_compras.head(10)

Tabla COMPRAS: 6,500 filas x 8 columnas


,id_cliente,fecha_compra,monto_compra,metodo_pago,tiempo,navegador,boletin,vale
0,504308,2021-02-02,109.0540,2,1022,0,True,False
1,504309,2021-10-10,35.3260,1,865,0,False,False
2,504310,2021-08-27,89.3530,2,900,2,True,True
3,504311,2021-06-10,27.2200,1,793,0,True,False
4,504312,2021-06-08,28.5310,0,729,0,False,False
5,504313,2021-11-10,16.6110,1,878,2,False,True
6,504314,2021-02-01,44.6610,1,807,1,True,False
7,504315,2021-09-12,54.9770,2,689,3,True,True
8,504316,2021-03-15,42.8390,1,525,0,False,True
9,504317,2021-02-04,43.8830,1,932,1,True,False


In [17]:
# ============================================================
# Verificación de integridad referencial
# Todos los id_cliente en compras deben existir en clientes
# ============================================================
ids_clientes = set(df_clientes['id_cliente'])
ids_compras = set(df_compras['id_cliente'])

huerfanos = ids_compras - ids_clientes
print(f"Clientes sin referencia en tabla compras (huérfanos): {len(huerfanos)}")

if len(huerfanos) == 0:
    print("Integridad referencial verificada: todos los id_cliente de compras existen en clientes.")
else:
    print(f"IDs huérfanos encontrados: {huerfanos}")

Clientes sin referencia en tabla compras (huérfanos): 0
Integridad referencial verificada: todos los id_cliente de compras existen en clientes.


---
## 7. Carga — Conexión a PostgreSQL en la nube (AWS)

Cargar los datos a una base de datos SQL en la nube.

In [18]:
# ============================================================
# CONFIGURACIÓN DE CONEXIÓN
# Ajustar estos valores a tu entorno AWS
# ============================================================
DB_USER     = 'postgres'              # Usuario de PostgreSQL
DB_PASSWORD = 'tu_contraseña'         # Contraseña
DB_HOST     = 'tu_ip_o_endpoint_aws'  # IP o endpoint del contenedor en AWS
DB_PORT     = '5432'                  # Puerto (5432 es el default)
DB_NAME     = 'ventas_online'         # Nombre de la base de datos

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Conectando a: postgresql://***@{DB_HOST}:{DB_PORT}/{DB_NAME}")

Conectando a: postgresql://***@18.117.240.51:5432/ventas_online


In [22]:
# ============================================================
# Crear engine de SQLAlchemy y probar conexión
# ============================================================
engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        version = result.scalar()
    print(f"Conexión exitosa")
    print(f"   PostgreSQL: {version}")
except Exception as e:
    print(f"Error de conexión: {e}")
    print("\nVerifica:")
    print("  1. Que el contenedor de PostgreSQL esté corriendo en AWS")
    print("  2. Que el host, puerto, usuario y contraseña sean correctos")
    print("  3. Que el security group/firewall permita conexiones en el puerto 5432")

Conexión exitosa
   PostgreSQL: PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


### 7.1 Verificar que las tablas existan

In [23]:
# Verificar que las tablas existen
inspector = inspect(engine)
tablas = inspector.get_table_names()
print(f"   Tablas en la base de datos: {tablas}")

   Tablas en la base de datos: ['clientes', 'compras']


### 7.2 Carga incremental con estrategia UPSERT (Insert + Update)

Esta función permite:
- Ejecutar el notebook múltiples veces sin duplicar datos (idempotente).
- Insertar automáticamente clientes y compras nuevos.
- Actualizar los registros existentes en la BD si sus datos han cambiado en el CSV.
- Omitir registros que permanezcan idénticos sin realizar escrituras innecesarias.

In [24]:
def cargar_incremental_upsert(df_clientes, df_compras, engine):
    """
    Carga incremental con estrategia UPSERT (Insert/Update):
    - Si el cliente o compra es nuevo, lo inserta en la BD.
    - Si ya existe pero sus datos cambiaron, actualiza el registro en la BD.
    - Si ya existe y no cambió nada, omite la operación.

    Args:
        df_clientes: DataFrame con columnas ['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras']
        df_compras:  DataFrame con columnas ['id_cliente', 'fecha_compra', 'monto_compra', 'metodo_pago',
                                             'tiempo', 'navegador', 'boletin', 'vale']
        engine:      SQLAlchemy engine conectado a PostgreSQL

    Returns:
        dict con estadísticas detalladas de la ejecución
    """
    stats = {
        'total_csv_clientes': len(df_clientes),
        'total_csv_compras': len(df_compras),
        'clientes_insertados': 0,
        'clientes_actualizados': 0,
        'clientes_sin_cambios': 0,
        'compras_insertadas': 0,
        'compras_actualizadas': 0,
        'compras_sin_cambios': 0,
    }

    with engine.begin() as conn:
        # ============================================================
        # 1. UPSERT TABLA CLIENTES
        # ============================================================
        df_exist_c = pd.read_sql("SELECT id_cliente, edad, genero, venta_total, n_compras FROM clientes", conn)

        if df_exist_c.empty:
            nuevos_c = df_clientes.copy()
            modificados_c = pd.DataFrame()
            sin_cambios_c_count = 0
        else:
            merged_c = pd.merge(df_clientes, df_exist_c, on='id_cliente', suffixes=('', '_exist'))

            cond_cambio_c = (
                (merged_c['edad'].astype(int) != merged_c['edad_exist'].astype(int)) |
                (merged_c['genero'].astype(int) != merged_c['genero_exist'].astype(int)) |
                ((merged_c['venta_total'].astype(float) - merged_c['venta_total_exist'].astype(float)).abs() > 1e-4) |
                (merged_c['n_compras'].astype(int) != merged_c['n_compras_exist'].astype(int))
            )

            modificados_c = merged_c[cond_cambio_c][['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras']]
            sin_cambios_c_count = len(merged_c[~cond_cambio_c])

            existentes_ids_c = set(df_exist_c['id_cliente'])
            nuevos_c = df_clientes[~df_clientes['id_cliente'].isin(existentes_ids_c)].copy()

        # Insertar clientes nuevos
        if not nuevos_c.empty:
            nuevos_c.to_sql('clientes', conn, if_exists='append', index=False)
            stats['clientes_insertados'] = len(nuevos_c)

        # Actualizar clientes modificados
        if not modificados_c.empty:
            for _, row in modificados_c.iterrows():
                conn.execute(text("""
                    UPDATE clientes
                    SET edad = :edad, genero = :genero, venta_total = :venta_total, n_compras = :n_compras
                    WHERE id_cliente = :id_cliente
                """), {
                    'id_cliente': int(row['id_cliente']),
                    'edad': int(row['edad']),
                    'genero': int(row['genero']),
                    'venta_total': float(row['venta_total']),
                    'n_compras': int(row['n_compras'])
                })
            stats['clientes_actualizados'] = len(modificados_c)

        stats['clientes_sin_cambios'] = sin_cambios_c_count

        # ============================================================
        # 2. UPSERT TABLA COMPRAS
        # ============================================================
        df_exist_p = pd.read_sql("""
            SELECT id_compra, id_cliente, fecha_compra, monto_compra, metodo_pago, tiempo, navegador, boletin, vale
            FROM compras
        """, conn)

        if df_exist_p.empty:
            nuevas_p = df_compras.copy()
            modificadas_p = pd.DataFrame()
            sin_cambios_p_count = 0
        else:
            df_exist_p['fecha_compra'] = pd.to_datetime(df_exist_p['fecha_compra'])

            merged_p = pd.merge(df_compras, df_exist_p, on='id_cliente', suffixes=('', '_exist'))

            cond_cambio_p = (
                (merged_p['fecha_compra'] != merged_p['fecha_compra_exist']) |
                ((merged_p['monto_compra'].astype(float) - merged_p['monto_compra_exist'].astype(float)).abs() > 1e-4) |
                (merged_p['metodo_pago'].astype(int) != merged_p['metodo_pago_exist'].astype(int)) |
                (merged_p['tiempo'].astype(int) != merged_p['tiempo_exist'].astype(int)) |
                (merged_p['navegador'].astype(int) != merged_p['navegador_exist'].astype(int)) |
                (merged_p['boletin'].astype(bool) != merged_p['boletin_exist'].astype(bool)) |
                (merged_p['vale'].astype(bool) != merged_p['vale_exist'].astype(bool))
            )

            modificadas_p = merged_p[cond_cambio_p][['id_compra', 'id_cliente', 'fecha_compra', 'monto_compra',
                                                   'metodo_pago', 'tiempo', 'navegador', 'boletin', 'vale']]
            sin_cambios_p_count = len(merged_p[~cond_cambio_p])

            existentes_ids_p = set(df_exist_p['id_cliente'])
            nuevas_p = df_compras[~df_compras['id_cliente'].isin(existentes_ids_p)].copy()

        # Insertar compras nuevas
        if not nuevas_p.empty:
            if 'id_compra' in nuevas_p.columns:
                nuevas_p = nuevas_p.drop(columns=['id_compra'])
            nuevas_p.to_sql('compras', conn, if_exists='append', index=False)
            stats['compras_insertadas'] = len(nuevas_p)

        # Actualizar compras modificadas
        if not modificadas_p.empty:
            for _, row in modificadas_p.iterrows():
                conn.execute(text("""
                    UPDATE compras
                    SET fecha_compra = :fecha_compra, monto_compra = :monto_compra, metodo_pago = :metodo_pago,
                        tiempo = :tiempo, navegador = :navegador, boletin = :boletin, vale = :vale
                    WHERE id_compra = :id_compra
                """), {
                    'id_compra': int(row['id_compra']),
                    'fecha_compra': row['fecha_compra'].strftime('%Y-%m-%d'),
                    'monto_compra': float(row['monto_compra']),
                    'metodo_pago': int(row['metodo_pago']),
                    'tiempo': int(row['tiempo']),
                    'navegador': int(row['navegador']),
                    'boletin': bool(row['boletin']),
                    'vale': bool(row['vale'])
                })
            stats['compras_actualizadas'] = len(modificadas_p)

        stats['compras_sin_cambios'] = sin_cambios_p_count

    with engine.connect() as conn:
        stats['total_clientes_bd'] = conn.execute(text("SELECT COUNT(*) FROM clientes")).scalar()
        stats['total_compras_bd'] = conn.execute(text("SELECT COUNT(*) FROM compras")).scalar()

    print("Proceso de Carga / Actualización completado exitosamente.")
    return stats


In [25]:
# ============================================================
# Ejecutar carga incremental con UPSERT
# ============================================================
print("Iniciando carga / actualización de datos...\n")
stats = cargar_incremental_upsert(df_clientes, df_compras, engine)

print("\n" + "="*60)
print("RESUMEN DE OPERACIÓN")
print("="*60)
print(f"  Clientes CSV procesados:   {stats['total_csv_clientes']:,}")
print(f"  Clientes Insertados:       {stats['clientes_insertados']:,}")
print(f"  Clientes Actualizados:     {stats['clientes_actualizados']:,}")
print(f"  Clientes Sin Cambios:      {stats['clientes_sin_cambios']:,}")
print("-" * 60)
print(f"  Compras CSV procesadas:    {stats['total_csv_compras']:,}")
print(f"  Compras Insertadas:        {stats['compras_insertadas']:,}")
print(f"  Compras Actualizadas:      {stats['compras_actualizadas']:,}")
print(f"  Compras Sin Cambios:       {stats['compras_sin_cambios']:,}")
print("-" * 60)
print(f"  TOTAL EN BD:               {stats['total_clientes_bd']:,} clientes | {stats['total_compras_bd']:,} compras")


Iniciando carga / actualización de datos...

Proceso de Carga / Actualización completado exitosamente.

RESUMEN DE OPERACIÓN
  Clientes CSV procesados:   6,500
  Clientes Insertados:       6,500
  Clientes Actualizados:     0
  Clientes Sin Cambios:      0
------------------------------------------------------------
  Compras CSV procesadas:    6,500
  Compras Insertadas:        6,500
  Compras Actualizadas:      0
  Compras Sin Cambios:       0
------------------------------------------------------------
  TOTAL EN BD:               6,500 clientes | 6,500 compras


---
## 8. Verificación final — Consultar la BD

Validamos que los datos cargados en PostgreSQL coincidan con el CSV original.

In [26]:
print("="*60)
print("8.1 VERIFICACIÓN DE CONTEOS E INTEGRIDAD")
print("="*60)

with engine.connect() as conn:
    n_clientes = conn.execute(text("SELECT COUNT(*) FROM clientes")).scalar()
    n_compras = conn.execute(text("SELECT COUNT(*) FROM compras")).scalar()

    ids_bd_clientes = set(pd.read_sql("SELECT id_cliente FROM clientes", conn)['id_cliente'])
    ids_bd_compras = set(pd.read_sql("SELECT DISTINCT id_cliente FROM compras", conn)['id_cliente'])

ids_csv = set(df_clientes['id_cliente'])

print(f"  Clientes en BD:        {n_clientes:,} (Clientes en CSV actual: {len(df_clientes):,})")
print(f"  Compras en BD:         {n_compras:,} (Compras en CSV actual: {len(df_compras):,})")

assert ids_csv.issubset(ids_bd_clientes), "ERROR: Existen clientes en el CSV que no se registraron en la tabla clientes"
assert ids_csv.issubset(ids_bd_compras), "ERROR: Existen clientes en el CSV que no tienen registro en la tabla compras"
assert n_clientes >= len(df_clientes), f"ERROR: Clientes en BD ({n_clientes}) es menor que en el CSV ({len(df_clientes)})"
assert n_compras >= len(df_compras), f"ERROR: Compras en BD ({n_compras}) es menor que en el CSV ({len(df_compras)})"

print("\nVerificación exitosa: Todos los registros del CSV están integrados y verificados en la BD.")


8.1 VERIFICACIÓN DE CONTEOS E INTEGRIDAD
  Clientes en BD:        6,500 (Clientes en CSV actual: 6,500)
  Compras en BD:         6,500 (Compras en CSV actual: 6,500)

Verificación exitosa: Todos los registros del CSV están integrados y verificados en la BD.


In [27]:
print("="*60)
print("8.2 MUESTRA DE DATOS — TABLA CLIENTES")
print("="*60)

with engine.connect() as conn:
    muestra_clientes = pd.read_sql(
        "SELECT * FROM clientes ORDER BY id_cliente LIMIT 10", conn
    )
muestra_clientes

8.2 MUESTRA DE DATOS — TABLA CLIENTES


,id_cliente,edad,genero,venta_total,n_compras
0,504308,18,1,737.4000,8
1,504309,46,1,689.2000,17
2,504310,38,0,95.6000,1
3,504311,44,1,75.7000,2
4,504312,41,0,46.0000,2
5,504313,24,1,20.9000,1
6,504314,18,1,269.0000,6
7,504315,27,0,376.5000,10
8,504316,43,0,122.3000,4
9,504317,44,0,239.4000,5


In [28]:
print("="*60)
print("8.3 MUESTRA DE DATOS — TABLA COMPRAS")
print("="*60)

with engine.connect() as conn:
    muestra_compras = pd.read_sql(
        "SELECT * FROM compras ORDER BY id_compra LIMIT 10", conn
    )
muestra_compras

8.3 MUESTRA DE DATOS — TABLA COMPRAS


,id_compra,id_cliente,fecha_compra,monto_compra,metodo_pago,tiempo,navegador,boletin,vale
0,1,504308,2021-02-02,109.0540,2,1022,0,True,False
1,2,504309,2021-10-10,35.3260,1,865,0,False,False
2,3,504310,2021-08-27,89.3530,2,900,2,True,True
3,4,504311,2021-06-10,27.2200,1,793,0,True,False
4,5,504312,2021-06-08,28.5310,0,729,0,False,False
5,6,504313,2021-11-10,16.6110,1,878,2,False,True
6,7,504314,2021-02-01,44.6610,1,807,1,True,False
7,8,504315,2021-09-12,54.9770,2,689,3,True,True
8,9,504316,2021-03-15,42.8390,1,525,0,False,True
9,10,504317,2021-02-04,43.8830,1,932,1,True,False


In [29]:
print("="*60)
print("8.4 VERIFICACIÓN JOIN — INTEGRIDAD REFERENCIAL EN BD")
print("="*60)

with engine.connect() as conn:
    resultado = pd.read_sql("""
        SELECT
            c.id_cliente, c.edad, c.genero, c.venta_total,
            p.fecha_compra, p.monto_compra, p.metodo_pago, p.navegador
        FROM clientes c
        JOIN compras p ON c.id_cliente = p.id_cliente
        ORDER BY c.id_cliente
        LIMIT 10
    """, conn)

print(f"JOIN exitoso: {len(resultado)} filas")
resultado

8.4 VERIFICACIÓN JOIN — INTEGRIDAD REFERENCIAL EN BD
JOIN exitoso: 10 filas


,id_cliente,edad,genero,venta_total,fecha_compra,monto_compra,metodo_pago,navegador
0,504308,18,1,737.4000,2021-02-02,109.0540,2,0
1,504309,46,1,689.2000,2021-10-10,35.3260,1,0
2,504310,38,0,95.6000,2021-08-27,89.3530,2,2
3,504311,44,1,75.7000,2021-06-10,27.2200,1,0
4,504312,41,0,46.0000,2021-06-08,28.5310,0,0
5,504313,24,1,20.9000,2021-11-10,16.6110,1,2
6,504314,18,1,269.0000,2021-02-01,44.6610,1,1
7,504315,27,0,376.5000,2021-09-12,54.9770,2,3
8,504316,43,0,122.3000,2021-03-15,42.8390,1,0
9,504317,44,0,239.4000,2021-02-04,43.8830,1,1


In [30]:
print("="*60)
print("8.5 VERIFICACIÓN RÁPIDA — DATOS MONETARIOS (4 decimales)")
print("="*60)

with engine.connect() as conn:
    monetario = pd.read_sql("""
        SELECT
            c.id_cliente,
            c.venta_total,
            p.monto_compra
        FROM clientes c
        JOIN compras p ON c.id_cliente = p.id_cliente
        ORDER BY c.id_cliente
        LIMIT 5
    """, conn)

print("Verificando que los campos monetarios mantengan precisión:")
print(monetario.to_string(index=False))
print("\nCampos monetarios con precisión NUMERIC(12,4) en la BD.")

8.5 VERIFICACIÓN RÁPIDA — DATOS MONETARIOS (4 decimales)
Verificando que los campos monetarios mantengan precisión:
 id_cliente  venta_total  monto_compra
     504308     737.4000      109.0540
     504309     689.2000       35.3260
     504310      95.6000       89.3530
     504311      75.7000       27.2200
     504312      46.0000       28.5310

Campos monetarios con precisión NUMERIC(12,4) en la BD.


In [31]:
engine.dispose()
print("Conexión con la BD cerrada.")

Conexión con la BD cerrada.
